# Домашнє завдання. Тема 3: Векторні операції для обробки векторів слів

In [51]:
!pip install -q plotly

import plotly.express as px
import numpy as np
import pandas as pd
import pickle
from sklearn.decomposition import PCA

In [52]:
!wget -q -O word_embeddings_subset.p https://github.com/faridjafarlee/numerical-programming-in-python-neoversity/raw/refs/heads/main/word_embeddings_subset.p

## 1. Визначте DataFrame з тривимірними векторами слів

In [53]:
with open('word_embeddings_subset.p', 'rb') as f:
    word_embeddings = pickle.load(f)

words = list(word_embeddings.keys())
vectors = np.array(list(word_embeddings.values()))

print(f"Кількість слів: {len(words)}")
print(f"Розмірність векторів: {vectors.shape[1]}")

Кількість слів: 243
Розмірність векторів: 300


In [54]:
pca = PCA(n_components=3)
vectors_3d = pca.fit_transform(vectors)

df = pd.DataFrame(vectors_3d, columns=['x', 'y', 'z'])
df['word'] = words
df = df[['word', 'x', 'y', 'z']]

df.head(20)

,word,x,y,z
0,country,0.746037,-0.387964,-0.482691
1,city,0.102492,0.140384,-1.189891
2,China,0.831055,-0.129500,-0.312599
3,Iraq,0.656337,-0.177791,-0.338381
4,oil,0.658345,-0.432464,-0.621213
5,town,0.378076,-0.080444,-1.053469
6,Canada,1.191961,-0.407709,-0.741816
7,London,-0.297986,0.253387,-1.151863
8,England,1.103978,-0.110078,-0.893429
9,Australia,1.049533,-0.452267,-0.835772


In [55]:
fig = px.scatter_3d(df, x='x', y='y', z='z', text='word', hover_name='word')
fig.update_traces(marker=dict(size=4), textposition='top center', textfont_size=7)
fig.update_layout(title='3D візуалізація word embeddings (PCA)', width=900, height=700)
fig.show()

## 2. Функція для пошуку найближчого слова

In [56]:
def find_closest_word(vector, df, exclude_word=None):
    coords = df[['x', 'y', 'z']].values.astype(float)
    distances = np.linalg.norm(coords - vector.astype(float), axis=1)
    if exclude_word:
        mask = df['word'] == exclude_word
        distances[mask] = np.inf
    closest_idx = np.argmin(distances)
    return df.iloc[closest_idx]['word'], distances[closest_idx]

In [57]:
test_words = ['happy', 'China', 'oil', 'queen', 'France', 'country']

for word in test_words:
    vector = df[df['word'] == word][['x', 'y', 'z']].values[0]
    closest, dist = find_closest_word(vector, df, exclude_word=word)
    print(f"Найближче до '{word}': '{closest}' (відстань: {dist:.4f})")

Найближче до 'happy': 'queen' (відстань: 0.1246)
Найближче до 'China': 'France' (відстань: 0.0907)
Найближче до 'oil': 'country' (відстань: 0.1699)
Найближче до 'queen': 'king' (відстань: 0.1026)
Найближче до 'France': 'China' (відстань: 0.0907)
Найближче до 'country': 'Thailand' (відстань: 0.1538)


## 3. Векторний добуток для знаходження ортогонального слова

In [67]:
pairs = [('happy', 'sad'), ('sad', 'happy'), ('China', 'Japan'), ('Japan', 'China'), ('oil', 'gas'), ('gas', 'oil'), ('London', 'England'), ('England', 'London')]

for word1, word2 in pairs:
    v1 = df[df['word'] == word1][['x', 'y', 'z']].values[0]
    v2 = df[df['word'] == word2][['x', 'y', 'z']].values[0]

    cross = np.cross(v1, v2)
    closest, dist = find_closest_word(cross, df)

    print(f"'{word1}' x '{word2}' -> найближче слово: '{closest}' (відстань: {dist:.4f})")

print(f"\n'happy' x 'sad' != 'sad' x 'happy' (Nuuk <> Qatar) - це тому що векторний добуток антикомутативний (a × b = −b × a)")

'happy' x 'sad' -> найближче слово: 'Nuuk' (відстань: 0.3786)
'sad' x 'happy' -> найближче слово: 'Qatar' (відстань: 0.5004)
'China' x 'Japan' -> найближче слово: 'Malta' (відстань: 0.5803)
'Japan' x 'China' -> найближче слово: 'Bahrain' (відстань: 0.5146)
'oil' x 'gas' -> найближче слово: 'Syria' (відстань: 0.2688)
'gas' x 'oil' -> найближче слово: 'Nuuk' (відстань: 0.0637)
'London' x 'England' -> найближче слово: 'Fiji' (відстань: 0.7937)
'England' x 'London' -> найближче слово: 'Zagreb' (відстань: 0.6549)

'happy' x 'sad' != 'sad' x 'happy' (Nuuk <> Qatar) - це тому що векторний добуток антикомутативний (a × b = −b × a)


## 4. Функція визначення кута між словами

In [59]:
def angle_between_words(word1, word2, df):
    v1 = df[df['word'] == word1][['x', 'y', 'z']].values[0]
    v2 = df[df['word'] == word2][['x', 'y', 'z']].values[0]

    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    cos_angle = np.clip(cos_angle, -1, 1)
    angle_rad = np.arccos(cos_angle)
    angle_deg = np.degrees(angle_rad)

    return angle_deg

In [60]:
test_pairs = [
    ('happy', 'sad'),
    ('China', 'Japan'),
    ('oil', 'gas'),
    ('London', 'England'),
    ('happy', 'oil'),
    ('France', 'Germany')
]

for word1, word2 in test_pairs:
    angle = angle_between_words(word1, word2, df)
    print(f"Кут між '{word1}' та '{word2}': {angle:.2f}°")

Кут між 'happy' та 'sad': 10.01°
Кут між 'China' та 'Japan': 16.02°
Кут між 'oil' та 'gas': 33.20°
Кут між 'London' та 'England': 67.17°
Кут між 'happy' та 'oil': 25.55°
Кут між 'France' та 'Germany': 23.41°


## Висновок

Слова з близьким значенням (наприклад, 'China' і 'Japan', 'oil' і 'gas') мають малий кут між векторами, що свідчить про їхню семантичну схожість. Слова з різних семантичних категорій (наприклад, 'happy' і 'oil') мають більший кут. Векторний добуток дає вектор, ортогональний до обох вхідних слів, і найближче до нього слово зазвичай не має прямого семантичного зв'язку з вихідною парою.